# BGE-M3 Dense Retrieval (GPU, Colab)

Same logic as `subtask1/scripts/bge_m3_retrieval.py` (embed evidence pool, embed claims, retrieve top-3 per claim by cosine similarity, score against gold evidence), just running on Colab's free GPU instead of local CPU -- encoding ~19,000 texts through a 568M-parameter model is much faster there.

Uses direct file upload/download (no Google Drive) -- faster for files this size. Self-contained: does not need the rest of the repo cloned, just the three data files uploaded directly in the browser (`train_subtask1.json`, `dev_subtask1.json`, `evidence_pool.json`).

After running, the last cell triggers a browser download of `bge_m3_results.json` and `bge_m3_summary.txt` directly -- drop them into `subtask1/results/bge_m3/` in the repo (rename to `results.json`/`summary.txt`).

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
!pip install -q -U sentence-transformers

### Upload the 3 data files
Running this cell opens a browser file picker -- select `train_subtask1.json`, `dev_subtask1.json`, and `evidence_pool.json` all at once (multi-select) from your computer. They upload directly into this Colab session, not Drive.

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

In [ ]:
TRAIN_FILE = "train_subtask1.json"
DEV_FILE = "dev_subtask1.json"
POOL_FILE = "evidence_pool.json"

OUT_RESULTS = "bge_m3_results.json"
OUT_SUMMARY = "bge_m3_summary.txt"

# Sanity check the upload cell above actually produced these exact filenames
import os
for f in [TRAIN_FILE, DEV_FILE, POOL_FILE]:
    assert os.path.exists(f), f"{f} not found -- re-run the upload cell and make sure all 3 files are selected"

In [ ]:
import json
import re

with open(TRAIN_FILE, encoding="utf-8") as f:
    train = json.load(f)
with open(DEV_FILE, encoding="utf-8") as f:
    dev = json.load(f)
with open(POOL_FILE, encoding="utf-8") as f:
    pool = json.load(f)

all_claims = train + dev
print(f"Loaded {len(all_claims)} claims ({len(train)} train + {len(dev)} dev), {len(pool)} evidence.")

### Small self-contained helpers
Exact same logic as `subtask1/data_loader.py` and `subtask1/evaluate.py` in the repo -- copied inline here so this notebook doesn't need the repo cloned, not reimplemented differently.

In [ ]:
import math

def detect_lang(text: str) -> str:
    bn = len(re.findall(r"[\u0980-\u09FF]", text))
    hi = len(re.findall(r"[\u0900-\u097F]", text))
    en = len(re.findall(r"[a-zA-Z]", text))
    scores = {"bn": bn * 2, "hi": hi * 2, "en": en}
    return max(scores, key=scores.get)

def is_code_mixed(text: str) -> bool:
    has_indic = bool(re.search(r"[\u0900-\u09FF]", text))
    has_latin = bool(re.search(r"[a-zA-Z]{4,}", text))
    return has_indic and has_latin

def build_pool_by_lang(pool: list) -> dict:
    by_lang = {}
    for item in pool:
        lang = detect_lang(item["Evidence"])
        by_lang.setdefault(lang, []).append(item)
    return by_lang

def build_pool_text_to_id(pool: list) -> dict:
    return {item["Evidence"]: item["ID"] for item in pool}

def success_at_k(retrieved_ids: list, gold_id: str, k: int = 3) -> int:
    return int(gold_id in retrieved_ids[:k])

def ndcg_at_k(retrieved_ids: list, gold_id: str, k: int = 3) -> float:
    for i, rid in enumerate(retrieved_ids[:k]):
        if rid == gold_id:
            return 1.0 / math.log2(i + 2)
    return 0.0

def evaluate_retrieval(results: list, pool_text_to_id: dict, k: int = 3) -> dict:
    s_at_k, ndcg = 0.0, 0.0
    n = len(results)
    for r in results:
        gold_id = pool_text_to_id[r["claim_item"]["Evidence"]]
        s_at_k += success_at_k(r["retrieved_ids"], gold_id, k)
        ndcg += ndcg_at_k(r["retrieved_ids"], gold_id, k)
    return {f"Success@{k}": s_at_k / n if n else 0.0,
            f"nDCG@{k}": ndcg / n if n else 0.0,
            "n": n}

In [ ]:
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading BAAI/bge-m3 on {device}...")
model = SentenceTransformer("BAAI/bge-m3", device=device)

In [ ]:
pool_texts_by_id = {p["ID"]: p["Evidence"] for p in pool}
claim_texts_by_id = {c["ID"]: c["Text"] for c in all_claims}

pool_ids = list(pool_texts_by_id.keys())
pool_texts = [pool_texts_by_id[i] for i in pool_ids]
print(f"Encoding {len(pool_texts)} evidence items...")
pool_vecs = model.encode(pool_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
pool_embeddings = dict(zip(pool_ids, pool_vecs))
print("Done encoding evidence pool.")

In [ ]:
claim_ids = list(claim_texts_by_id.keys())
claim_texts = [claim_texts_by_id[i] for i in claim_ids]
print(f"Encoding {len(claim_texts)} claims...")
claim_vecs = model.encode(claim_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
claim_embeddings = dict(zip(claim_ids, claim_vecs))
print("Done encoding claims.")

In [ ]:
import numpy as np

pool_by_lang = build_pool_by_lang(pool)
lang_ids, lang_matrix = {}, {}
for lang, items in pool_by_lang.items():
    ids = [item["ID"] for item in items]
    lang_ids[lang] = ids
    lang_matrix[lang] = np.array([pool_embeddings[i] for i in ids])

gold_evidence_by_id = {c["ID"]: c["Evidence"] for c in all_claims if "Evidence" in c}
pool_text_to_id = build_pool_text_to_id(pool)

BUCKET_LABELS = {"hi": "Hindi", "bn": "Bengali", "en": "English", "code-mixed": "Code-mixed"}
output_rows = []
retrieval_by_bucket = {"hi": [], "bn": [], "en": [], "code-mixed": []}

for c in all_claims:
    claim_text = c["Text"]
    lang = detect_lang(claim_text)
    query_vec = claim_embeddings[c["ID"]]
    ids = lang_ids.get(lang, [])
    matrix = lang_matrix.get(lang)
    if matrix is None or len(ids) == 0:
        continue
    sims = matrix @ query_vec
    top_idx = np.argsort(-sims)[:3]
    top3 = [(ids[i], float(sims[i])) for i in top_idx]

    output_rows.append({
        "ID": c["ID"],
        "Claim": claim_text,
        "top_3_evidence": [
            {"ID": eid, "Evidence": pool_texts_by_id.get(eid, ""), "score": score} for eid, score in top3
        ],
    })

    gold_text = gold_evidence_by_id.get(c["ID"])
    if gold_text:
        bucket = "code-mixed" if is_code_mixed(claim_text) else lang
        retrieval_by_bucket.setdefault(bucket, [])
        retrieval_by_bucket[bucket].append(
            {"claim_item": {"Evidence": gold_text}, "retrieved_ids": [eid for eid, _ in top3]}
        )

print(f"Retrieved top-3 for {len(output_rows)} claims.")

In [ ]:
with open(OUT_RESULTS, "w", encoding="utf-8") as f:
    json.dump(output_rows, f, ensure_ascii=False, indent=2)
print(f"Wrote {len(output_rows)} rows -> {OUT_RESULTS}")

all_results = [r for bucket_results in retrieval_by_bucket.values() for r in bucket_results]
overall = evaluate_retrieval(all_results, pool_text_to_id, k=3)

summary_lines = [
    "=== BGE-M3 Dense retrieval (data/ train+dev, Colab GPU) ===",
    f"{'Language':<12} {'Success@3':<12} {'nDCG@3':<12} {'n':<8}",
    "-" * 44,
]
for key in ["hi", "bn", "en", "code-mixed"]:
    if retrieval_by_bucket[key]:
        m = evaluate_retrieval(retrieval_by_bucket[key], pool_text_to_id, k=3)
        summary_lines.append(f"{BUCKET_LABELS[key]:<12} {m['Success@3']:<12.3f} {m['nDCG@3']:<12.3f} {m['n']:<8}")
summary_lines.append(f"{'Overall':<12} {overall['Success@3']:<12.3f} {overall['nDCG@3']:<12.3f} {overall['n']:<8}")
summary_text = "\n".join(summary_lines)
print("\n" + summary_text)

with open(OUT_SUMMARY, "w", encoding="utf-8") as f:
    f.write(summary_text + "\n")
print(f"\nWrote summary -> {OUT_SUMMARY}")

### Download the results directly (no Drive)

In [ ]:
from google.colab import files
files.download(OUT_RESULTS)
files.download(OUT_SUMMARY)